In [4]:
import pandas as pd

# Getting the Scoring Sequence for each divison

In [5]:
divisions = {
    "AL-East":    ("BAL", "BOS", "NYA", "TBA", "TOR"),
    "AL-Central": ("CHA", "CLE", "DET", "KCA", "MIN"),
    "AL-West":    ("ANA", "HOU", "OAK", "SEA", "TEX"),

    "NL-East":    ("ATL", "MIA", "NYN", "PHI", "WAS"),
    "NL-Central": ("CHN", "CIN", "MIL", "PIT", "SLN"),
    "NL-West":    ("ARI", "COL", "LAN", "SDN", "SFN")
}

rows = []

for season in range(2014, 2025):
    for division, teams in divisions.items():
        rows.append({
            "season": season,
            "league": "MLB",
            "division": division,
            "division_teams": teams
        })

division_df = pd.DataFrame(rows)
# division_df.to_csv("mlb_divisions.csv", index=False)

games_df = pd.read_csv("../Data/game_data_us_leagues.csv")

# Make one row per season/team/division
team_division_df = (
    division_df
    .explode("division_teams")
    .rename(columns={"division_teams": "team"})
)

# this is only the baseball games
mlb_games = games_df[
    (games_df["league"] == "MLB") &
    (games_df["season"].between(2014, 2024))
].copy()

# adding team 1 and team 2 division by merging with our mlb_divisions
mlb_games = mlb_games.merge(
    team_division_df[["season", "team", "division"]],
    left_on=["season", "team1"],
    right_on=["season", "team"],
    how="left"
)

mlb_games = (
    mlb_games
    .rename(columns={"division": "division1"})
    .drop(columns="team")
)

# adds team 2 div
mlb_games = mlb_games.merge(
    team_division_df[["season", "team", "division"]],
    left_on=["season", "team2"],
    right_on=["season", "team"],
    how="left"
)

mlb_games = (
    mlb_games
    .rename(columns={"division": "division2"})
    .drop(columns="team")
)

# season team1 team2 division1 division2

# FILTERING BY DIVISON
div_games = mlb_games[
    mlb_games["division1"] == mlb_games["division2"]
].copy()

div_games["division"] = div_games["division1"]

div_games = div_games.drop(columns=["division1", "division2"])

# go game by game and figure out the winner
def get_winner(row):
    if row["score1"] > row["score2"]:
        return row["team1"]
    elif row["score2"] > row["score1"]:
        return row["team2"]
    else:
        return None


div_games["winner"] = div_games.apply(
    get_winner,
    axis=1
)

# pairs need to have same ordering so wehn we do pd.group we dont double count
teams = div_games[["team1", "team2"]]

div_games["team1"] = teams.min(axis=1)
div_games["team2"] = teams.max(axis=1)

# mark whoever winner in a div game
div_games["win1"] = (
    div_games["winner"] == div_games["team1"]
).astype(int)

div_games["win2"] = (
    div_games["winner"] == div_games["team2"]
).astype(int)

# compute the series games
series_df = (
    div_games
    .groupby(
        ["season", "division", "team1", "team2"],
        as_index=False
    )
    .agg(
        wins1=("win1", "sum"),
        wins2=("win2", "sum")
    )
)

# determine the full series winner
def get_series_result(row):
    if row["wins1"] > row["wins2"]:
        return 1
    elif row["wins2"] > row["wins1"]:
        return -1
    else:
        return 0


series_df["result"] = series_df.apply(
    get_series_result,
    axis=1
)

# adds the other column we need to specifiy that this is mlb
series_df.insert(
    1,
    "league",
    "MLB"
)

series_df = series_df[
    [
        "season",
        "league",
        "division",
        "team1",
        "team2",
        "result",
        "wins1",
        "wins2"
    ]
]

# GET THE SERIES WINS FOR EACH TEAM

team1 = series_df[
    ["season", "league", "division", "team1", "result"]
].copy()

team1 = team1.rename(columns={"team1": "team"})

team1["wins"] = (
    team1["result"] == 1
).astype(int)

team2 = series_df[
    ["season", "league", "division", "team2", "result"]
].copy()

team2 = team2.rename(columns={"team2": "team"})

team2["wins"] = (
    team2["result"] == -1
).astype(int)

# create a ranking
score_sequences = pd.concat(
    [team1, team2],
    ignore_index=True
)

score_sequences = (
    score_sequences
    .groupby(
        ["season", "league", "division", "team"],
        as_index=False
    )["wins"]
    .sum()
)


score_sequences = score_sequences.sort_values(
    ["season", "division", "wins"]
)

In [6]:
score_sequences.to_csv('output.csv', index=False)

# Getting upset frequencies per division

In [7]:
# lookup each team's score for a given season and division
score_lookup = score_sequences.set_index(
    ["season", "division", "team"]
)["wins"].to_dict()


# determine whether each series is an upset
def is_upset(row):
    if row["result"] == 0:
        return 0

    if row["result"] == 1:
        winner = row["team1"]
        loser = row["team2"]
    else:
        winner = row["team2"]
        loser = row["team1"]

    winner_score = score_lookup[
        (row["season"], row["division"], winner)
    ]

    loser_score = score_lookup[
        (row["season"], row["division"], loser)
    ]

    return int(winner_score < loser_score)


series_df["upset"] = series_df.apply(
    is_upset,
    axis=1
)


# count upsets for each season and division
upset_counts = (
    series_df
    .groupby(
        ["season", "league", "division"],
        as_index=False
    )["upset"]
    .sum()
)

In [8]:
upset_counts.to_csv('output.csv', index=False)

# Merging The DFs

In [9]:
# sort teams by score within each season/division
score_sequences = score_sequences.sort_values(
    ["season", "division", "wins"],
    ascending=[True, True, False]
)

# collapse scoring sequence into one row per season/division
scoring_summary = (
    score_sequences
    .groupby(
        ["season", "league", "division"],
        as_index=False
    )
    .agg(
        num_teams=("team", "size"),
        teams=("team", tuple),
        scoring_sequence=("wins", tuple)
    )
)

# get number of in-division games played between each pair
games_per_pair = (
    div_games
    .groupby(
        ["season", "division", "team1", "team2"]
    )
    .size()
    .reset_index(name="games_per_pair")
    .groupby(
        ["season", "division"],
        as_index=False
    )["games_per_pair"]
    .first()
)

# merge scoring sequences, games per pair, and upset counts
analysis_df = (
    scoring_summary
    .merge(
        games_per_pair,
        on=["season", "division"],
        how="left"
    )
    .merge(
        upset_counts,
        on=["season", "league", "division"],
        how="left"
    )
)

# final column order
analysis_df = analysis_df[
    [
        "season",
        "division",
        "num_teams",
        "games_per_pair",
        "teams",
        "scoring_sequence",
        "upset"
    ]
]

In [11]:
analysis_df.to_csv(
    "../results/mlb/mlb_analysis.csv",
    index=False
)